# Evaluating a RAG Pipeline with DeepEval

A RAG app can fail in two places: the **retriever** (wrong or missing context) or the
**generator** (a good LLM answer that still ignores the context). DeepEval gives each surface its
own metric:

| Stage | Metric | Question it answers |
| --- | --- | --- |
| Retriever | `ContextualRelevancyMetric` | Is the retrieved context relevant to the question? |
| Retriever | `ContextualPrecisionMetric` | Is the relevant context ranked above the irrelevant? |
| Retriever | `ContextualRecallMetric` | Does the context contain everything the answer needs? |
| Generator | `FaithfulnessMetric` | Does the answer stick to the retrieved context (no hallucination)? |
| Generator | `AnswerRelevancyMetric` | Does the answer actually address the question? |

We'll build a deliberately tiny RAG pipeline, run it on a few questions, and score it with all five.

## 1. Install packages

We only need three libraries: `deepeval` itself, `google-genai` (so DeepEval's judge can talk to
Gemini), and `groq` (the model we're evaluating). No OpenAI key anywhere in this notebook.

In [1]:
%pip install -q deepeval google-genai groq


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 2. Add your API keys

Two roles, kept separate the whole way through this course:

- **Groq** — the *model under test*. It free-tier serves `llama-3.3-70b-versatile`. Get a key at
  https://console.groq.com/keys
- **Gemini** — the *judge*. Every DeepEval metric needs an LLM to grade with, and we use Gemini so
  you never need an OpenAI key. Get a free key at https://aistudio.google.com/apikey

Keys are entered with `getpass` so they never get printed or saved into the notebook file.

In [2]:
import os, getpass

os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")  # skip DeepEval's anonymous telemetry

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your GEMINI_API_KEY: ")
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]  # some libs read this name instead

print("Keys set.")

Keys set.


## 3. A tiny knowledge base + a naive retriever

We keep retrieval intentionally simple — plain keyword overlap, no embeddings, no vector database
— so the notebook stays focused on **evaluating** a RAG pipeline rather than building one. Swap
this for your real retriever and everything below still works unchanged.

In [3]:
CORPUS = [
    {"title": "Tokens", "content": (
        "Large language models split text into tokens, common character sequences roughly 4 "
        "characters or three-quarters of a word long. Both the prompt and the output are counted "
        "in tokens, and pricing and context limits are measured in tokens.")},
    {"title": "Embeddings", "content": (
        "An embedding is a fixed-length vector that represents the meaning of a piece of text. "
        "Texts with similar meaning have vectors that are close together, usually measured by "
        "cosine similarity, which is what lets a system do semantic search.")},
    {"title": "Retrieval-Augmented Generation", "content": (
        "RAG grounds an LLM's answer in external documents. At query time the system retrieves "
        "the most relevant chunks and passes them to the model as context, which reduces "
        "hallucination and lets you update knowledge without retraining.")},
    {"title": "Hallucination", "content": (
        "A hallucination is fluent, confident text that is factually wrong or unsupported by its "
        "sources. It happens because the model predicts likely text rather than looking facts up. "
        "Grounding answers in retrieved context is the main defense.")},
    {"title": "AI agents", "content": (
        "An AI agent is an LLM given a goal, tools, and a loop: plan, call a tool, observe the "
        "result, decide the next step, until the task is done.")},
]


def retrieve(query: str, k: int = 2) -> list[str]:
    """Naive retriever: rank docs by keyword overlap with the query. Good enough to teach evals."""
    q_words = set(query.lower().split())
    scored = []
    for doc in CORPUS:
        doc_words = set((doc["title"] + " " + doc["content"]).lower().split())
        overlap = len(q_words & doc_words)
        scored.append((overlap, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc["content"] for _, doc in scored[:k]]

## 4. Generate an answer with Groq (the model under test)

This is the RAG app itself: retrieved passages go into the prompt, Groq writes the answer.

In [4]:
from groq import Groq

groq_client = Groq()

RAG_SYSTEM = (
    "Answer ONLY using the provided context. If the context doesn't contain the answer, say you "
    "don't have that information. Be concise (1-2 sentences)."
)


def answer_with_groq(question: str, passages: list[str]) -> str:
    context = "\n\n".join(passages)
    prompt = f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    resp = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "system", "content": RAG_SYSTEM}, {"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content.strip()

## 5. Run the pipeline on a few questions

Two questions the knowledge base actually covers, and one it doesn't — watch what happens to that
one once we score it.

In [5]:
QUESTIONS = [
    {"input": "What is a token in an LLM?",
     "expected_output": "A token is a common character sequence, roughly 4 characters, used to measure both prompt and output length."},
    {"input": "How does RAG reduce hallucination?",
     "expected_output": "RAG retrieves relevant chunks and passes them as context, grounding the answer in real documents instead of only the model's training."},
    {"input": "How do I containerize a model for deployment with Docker?"},  # not in our tiny KB
]

rag_rows = []
for q in QUESTIONS:
    passages = retrieve(q["input"])
    answer = answer_with_groq(q["input"], passages)
    rag_rows.append({**q, "actual_output": answer, "retrieval_context": passages})
    print("Q:", q["input"])
    print("A:", answer)
    print("-" * 80)

Q: What is a token in an LLM?
A: I don't have that information. The provided context does not mention what a token in an LLM is.
--------------------------------------------------------------------------------


Q: How does RAG reduce hallucination?
A: RAG reduces hallucination by grounding an LLM's answer in external documents, providing the model with relevant context at query time. This approach lets the model base its answers on actual information from the documents, rather than generating potentially inaccurate information.
--------------------------------------------------------------------------------


Q: How do I containerize a model for deployment with Docker?
A: I don't have that information. The provided context only discusses embeddings, semantic search, and hallucinations in text models, but does not mention Docker or model deployment.
--------------------------------------------------------------------------------


## 6. Turn each result into an `LLMTestCase`

An `LLMTestCase` is what a metric actually reads: the question, what your app answered, what it
*should* have answered (if you have a reference), and what context was retrieved.

In [6]:
from deepeval.test_case import LLMTestCase

test_cases = [
    LLMTestCase(
        input=r["input"],
        actual_output=r["actual_output"],
        expected_output=r.get("expected_output"),
        retrieval_context=r["retrieval_context"],
    )
    for r in rag_rows
]

## The judge: `GeminiModel`

Every DeepEval metric defaults to OpenAI if you don't tell it otherwise — even a metric that
scores deterministically will still try to build an OpenAI client unless you pass `model=`. We
build **one Gemini judge** here and pass it to every metric in this notebook.

In [7]:
from deepeval.models import GeminiModel

judge = GeminiModel(model="gemini-2.5-flash", api_key=os.environ["GEMINI_API_KEY"], temperature=0)
print("Judge ready:", judge.get_model_name())

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Judge ready: gemini-2.5-flash (Gemini)


## 8. Define the 5 RAG metrics

Every metric gets `model=judge` — Gemini grades, Groq answers.

In [8]:
from deepeval.metrics import (
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
    FaithfulnessMetric,
)

metrics = [
    FaithfulnessMetric(model=judge),
    AnswerRelevancyMetric(model=judge),
    ContextualRelevancyMetric(model=judge),
    ContextualPrecisionMetric(model=judge),
    ContextualRecallMetric(model=judge),
]

## 9. Run the evaluation

`ContextualPrecisionMetric` and `ContextualRecallMetric` need `expected_output` — our third
question doesn't have one, so DeepEval simply skips those two metrics for that row instead of
crashing (`skip_on_missing_params=True`).

In [9]:
from deepeval import evaluate
from deepeval.evaluate.configs import AsyncConfig, DisplayConfig, ErrorConfig

results = evaluate(
    test_cases=test_cases,
    metrics=metrics,
    async_config=AsyncConfig(max_concurrent=2),          # be gentle on free-tier rate limits
    display_config=DisplayConfig(print_results=False),
    error_config=ErrorConfig(ignore_errors=True, skip_on_missing_params=True),
)

✨ You're running DeepEval's latest Faithfulness Metric! (using gemini-2.5-flash (Gemini), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gemini-2.5-flash (Gemini), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gemini-2.5-flash (Gemini), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using gemini-2.5-flash (Gemini), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gemini-2.5-flash (Gemini), strict=False, 
async_mode=True)...

...

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/opt/miniconda3/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x10848c5c0> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/opt/miniconda3/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x10848c5c0> is already entered

⚠ WARNING: All metrics errored across every test case — no metric scores were recorded. Posting the run anyway so 
you can inspect the trace + span errors on the Confident AI dashboard.

⚠ WARNING: No hyperparameters logged.
» ]8;id=635314;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 5.65s | token cost: 0.0008353499999999999 USD)
» Test Results (3 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 3

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

## Read the results

`evaluate()` returns an `EvaluationResult` with one `TestResult` per test case. Each `TestResult`
carries a `metrics_data` list — one entry per metric, with a `score`, a `reason` (the judge's
explanation), and whether it `success`-fully passed its threshold.

If a metric's judge call itself fails (a rate limit, an auth error, a transient API error), the
metric has **no score** — `score` is `None`, and the real explanation lives in `.error` instead of
`.reason`. Always guard for that instead of formatting `.score` directly, or you'll get a
`TypeError: unsupported format string passed to NoneType.__format__` in place of the real error.

If you see `⚠ WARNING: All metrics errored across every test case`, every metric hit the same
failure — almost always an invalid/expired `GEMINI_API_KEY`, a Gemini free-tier rate limit (5
metrics x N test cases fires a burst of judge calls), or a package version mismatch in whichever
environment/kernel you're running (re-check `pip show deepeval google-genai`). Print
`results.test_results[0].metrics_data[0].error` to see the real exception immediately.

In [10]:
for tr in results.test_results:
    print("=" * 90)
    print("INPUT:", tr.input)
    print("ACTUAL OUTPUT:", tr.actual_output)
    for md_ in tr.metrics_data:
        verdict = "PASS" if md_.success else "FAIL"
        score = f"{md_.score:.2f}" if md_.score is not None else "ERRORED"
        print(f"  [{verdict}] {md_.name}: {score}")
        print(f"      reason: {md_.reason or md_.error}")

INPUT: What is a token in an LLM?
ACTUAL OUTPUT: I don't have that information. The provided context does not mention what a token in an LLM is.
  [FAIL] Faithfulness: ERRORED
      reason: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 31.364947338s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.c

## Takeaway

Watch the third question (outside our tiny knowledge base): `ContextualRelevancyMetric` and
`FaithfulnessMetric` should drop, because the retriever fetched irrelevant passages and the model
either says "I don't know" or drifts. **That's the signal.** A low RAG-metric score almost always
means *fix the retriever or the knowledge base*, not the LLM — which is exactly why DeepEval splits
retriever metrics from generator metrics instead of giving you one fuzzy "quality" number.

Next: [`02_agent_evals_with_deepeval.ipynb`](02_agent_evals_with_deepeval.ipynb) — evaluating an
agent that chooses tools instead of just answering.